# 📖 Notebook 1: Code Submission & Execution Pipeline

When you click **"Submit"** on LeetCode, a lot happens behind the scenes.
Your code doesn’t just run instantly — it goes through a carefully designed
pipeline that handles thousands of concurrent submissions reliably.

This notebook walks through the **full journey of a code submission** — from
browsing a problem to getting your verdict back.

---

## 🎯 Learning Objectives

By the end of this notebook, you will understand:

1. **The submission lifecycle** — how a submission moves through statuses: `pending → running → accepted / wrong_answer / ...`
2. **Why async processing is needed** — why the API server can’t just run your code directly
3. **How queues decouple API from workers** — the key pattern that makes this scalable
4. **The polling pattern** — how the frontend knows when your results are ready (spoiler: it asks repeatedly!)

---

**Notebooks in this series:**
- **📖 Notebook 1: Code Submission & Execution Pipeline** ← you are here
- 📖 Notebook 2: Sandboxed Code Execution (coming next)


## 🛠️ Setup

### 1. Start the infrastructure

Open a terminal and run:

```bash
cd system-designs/leetcode
docker-compose up -d
```

This starts:
- **PostgreSQL** — stores problems, users, submissions
- **Redis** — will be used for caching and pub/sub in later notebooks
- **Sandbox container** — isolated environment for running untrusted code (Notebook 2)
- **Adminer** — web UI for exploring the database
- **RedisInsight** — web UI for exploring Redis

### 2. Visualization tools

| Tool | URL | Login |
|------|-----|-------|
| Adminer | [http://localhost:8080](http://localhost:8080) | System: PostgreSQL, Server: `postgres`, User: `demo`, Password: `demo`, Database: `leetcode_demo` |
| RedisInsight | [http://localhost:5540](http://localhost:5540) | Add database → host: `host.docker.internal`, port: `6379` |

### 3. Select the notebook kernel

Make sure you’re using the **`.venv`** kernel:

1. Create the virtual environment (if you haven’t already):
   ```bash
   cd system-designs/leetcode
   uv venv
   source .venv/bin/activate
   uv sync
   ```
2. In VS Code, click the **kernel picker** (top-right of the notebook)
3. Select **`.venv (Python 3.x)`**
4. If it doesn’t appear, reload VS Code (`Cmd+Shift+P` → `Reload Window`)


In [ ]:
# === 🔌 Connection Setup ===

import psycopg2
import psycopg2.extras
import redis
import time
import json
import threading
import queue as queue_module  # Python standard library — thread-safe queue

# --- Database configuration ---
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "leetcode_demo",
    "user": "demo",
    "password": "demo",
}

# --- Redis configuration ---
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True,
}


def get_db_connection():
    """Create a new PostgreSQL connection."""
    return psycopg2.connect(**DB_CONFIG)


def get_redis_client():
    """Create a new Redis client."""
    return redis.Redis(**REDIS_CONFIG)


# --- Test connections ---
try:
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM problems")
    count = cur.fetchone()[0]
    print(f"✅ PostgreSQL connected — {count} problems in database")
    cur.close()
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL connection failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis connection failed: {e}")


## 📋 Step 1: Browsing Problems

The first thing a user does on LeetCode is **browse the problem list**.

Behind the scenes, this is a simple database query. The API endpoint
`GET /problems` fetches rows from the `problems` table, with pagination
so we don’t load all 3,000+ problems at once.

Let’s see what’s in our database:


In [ ]:
# === 📋 List All Problems (GET /problems) ===

conn = get_db_connection()
cur = conn.cursor()

# In production, you'd add pagination: LIMIT 20 OFFSET 0, LIMIT 20 OFFSET 20, etc.
cur.execute("""
    SELECT id, title, difficulty, tags
    FROM problems
    ORDER BY id
    LIMIT 10 OFFSET 0
""")

problems = cur.fetchall()

print(f"{'ID':<4} {'Title':<45} {'Difficulty':<10} {'Tags'}")
print("-" * 90)
for p in problems:
    tags = ", ".join(p[3]) if p[3] else ""
    print(f"{p[0]:<4} {p[1]:<45} {p[2]:<10} {tags}")

cur.close()
conn.close()


In [ ]:
# === 🔍 View a Single Problem (GET /problems/1) ===

conn = get_db_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("SELECT * FROM problems WHERE id = %s", (1,))
problem = cur.fetchone()

print(f"📌 Problem #{problem['id']}: {problem['title']}")
print(f"   Difficulty: {problem['difficulty']}")
print(f"   Tags: {', '.join(problem['tags'])}")
print(f"\n📝 Description:")
print(f"   {problem['description']}")
print(f"\n💻 Code Stub (Python):")
stubs = problem['code_stubs']
for line in stubs['python'].split('\n'):
    print(f"   {line}")
print(f"\n🧪 Test Cases:")
for i, tc in enumerate(problem['test_cases']):
    print(f"   Test {i+1}: Input={tc['input']} → Expected={tc['expected']}")

cur.close()
conn.close()


## 📝 Step 2: Writing & Submitting Code

The user writes their solution in the browser (LeetCode uses the **Monaco editor** — the
same editor that powers VS Code). When they click **Submit**, the browser sends a
`POST /problems/:id/submit` request with the code.

What does the API server do? It creates a **submission record** in the database
with `status = 'pending'`. That’s it — it does NOT run the code itself.

Let’s see this in action:


In [ ]:
# === 📝 Creating a Submission (POST /problems/1/submit) ===
#
# This is what the API server does when a user clicks "Submit".
# It just creates a database record — it does NOT run the code.

user_code = """
class Solution:
    def twoSum(self, nums, target):
        seen = {}
        for i, num in enumerate(nums):
            complement = target - num
            if complement in seen:
                return [seen[complement], i]
            seen[num] = i
"""

conn = get_db_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    INSERT INTO submissions (user_id, problem_id, language, code, status)
    VALUES (%s, %s, %s, %s, 'pending')
    RETURNING id, user_id, problem_id, language, status, created_at
""", (1, 1, 'python', user_code))

submission = cur.fetchone()
conn.commit()

print("✅ Submission created!")
print(f"   ID:       {submission['id']}")
print(f"   User:     {submission['user_id']}")
print(f"   Problem:  {submission['problem_id']}")
print(f"   Status:   {submission['status']}  ← notice it's 'pending'")
print(f"   Created:  {submission['created_at']}")

# Save this ID — we'll track submissions created in this notebook for cleanup
manual_submission_id = submission['id']
notebook_submission_ids = [manual_submission_id]

cur.close()
conn.close()


### 🤔 Why not just run the code right here?

You might wonder: why doesn’t the API server just execute the code and return
the results in the same HTTP response?

Here’s why:

1. **Code execution is slow** — solutions can take 1–5 seconds to run. During that
   time, the API server thread is **blocked** and can’t serve other requests.

2. **Timeouts** — HTTP requests typically have a 30-second timeout. What if the user’s
   code has an infinite loop? The entire request would hang.

3. **Resource isolation** — running untrusted code on the API server is a massive
   security risk. Code needs to run in an **isolated sandbox** (Notebook 2).

4. **Scaling** — during a contest, thousands of submissions arrive per second. The API
   server can’t run them all at once.

This is a classic example of the **Long-Running Tasks pattern**: the API accepts the
request, saves it, and tells the client "here’s your submission ID, come back later
for the results."


## ⚡ Step 3: The Queue — Decoupling API from Workers

The solution is a **message queue** sitting between the API and the workers.
Here’s the architecture:

```
                              ┌─────────────┐
  Client ──→ API Server ──→  │    Queue     │  ──→ Worker ──→ Sandbox
               │              └─────────────┘        │
               │                                     │
               ▼                                     ▼
        "Here's your                          Updates DB with
         submission ID,                       results (accepted,
         poll for results"                    wrong_answer, etc.)
```

**Why a queue?**

- **Buffering** — if 1,000 submissions arrive in one second, the queue holds them
  until workers can process them. Nobody gets an error.
- **Decoupling** — the API server doesn’t need to know anything about how code
  execution works. It just puts a message on the queue.
- **Scaling** — need to process submissions faster? Just add more workers.
  No changes to the API server needed.
- **Retry** — if a worker crashes mid-execution, the message stays in the queue
  and another worker can pick it up.

In production, companies use **SQS** (AWS), **RabbitMQ**, or **Redis Streams**.
For this demo, we’ll use Python’s built-in `queue.Queue` — same concept, simpler setup.


In [ ]:
# === ⚡ The Queue and Submit Function ===
#
# In production: Amazon SQS, RabbitMQ, or Redis Streams
# For this demo: Python's built-in thread-safe queue (same concept!)

job_queue = queue_module.Queue()


def submit_code(user_id, problem_id, language, code):
    """
    Simulates what the API server does when a user clicks Submit:
    1. Create a submission record in the DB (status='pending')
    2. Put the job on the queue for a worker to pick up
    3. Return the submission ID immediately — don't wait!
    """
    conn = get_db_connection()
    cur = conn.cursor()

    # Step 1: Create the submission record
    cur.execute("""
        INSERT INTO submissions (user_id, problem_id, language, code, status)
        VALUES (%s, %s, %s, %s, 'pending')
        RETURNING id
    """, (user_id, problem_id, language, code))
    submission_id = cur.fetchone()[0]
    conn.commit()
    cur.close()
    conn.close()

    # Step 2: Put the job on the queue
    job = {
        "submission_id": submission_id,
        "problem_id": problem_id,
        "language": language,
        "code": code,
    }
    job_queue.put(job)

    timestamp = time.strftime('%H:%M:%S')
    print(f"[API] 📨 Submission {submission_id} created and queued at {timestamp}")

    # Step 3: Return immediately — the client will poll for results
    return submission_id


print("✅ Queue and submit_code() function ready")


In [ ]:
# === ⚙️ The Worker — Processes Submissions from the Queue ===

worker_running = True


def run_test_case(code, test_case):
    """
    Run user code against a single test case.

    ⚠️ WARNING: Using exec() here is ONLY for demo purposes!
    In production, code runs in an isolated sandbox container.
    Notebook 2 covers real sandboxing in detail.
    """
    try:
        namespace = {}
        exec(code, namespace)

        solution = namespace['Solution']()
        input_data = test_case['input']

        # Call twoSum (this demo focuses on Two Sum — problem #1)
        result = solution.twoSum(input_data['nums'], input_data['target'])

        # Compare results (sort both — index order may vary)
        expected = test_case['expected']
        passed = sorted(result) == sorted(expected)
        return {"passed": passed, "output": result, "expected": expected}

    except Exception as e:
        return {"passed": False, "error": str(e)}


def worker():
    """
    The worker loop — runs forever in a background thread:
    1. Pull a job from the queue (blocks until one is available)
    2. Update submission status to 'running'
    3. Execute the code against each test case
    4. Update the database with final status and results
    """
    while worker_running:
        try:
            # Wait up to 1 second for a job, then check if we should stop
            try:
                job = job_queue.get(timeout=1)
            except queue_module.Empty:
                continue

            submission_id = job["submission_id"]
            timestamp = time.strftime('%H:%M:%S')
            print(f"[Worker] 🔧 Processing submission {submission_id} at {timestamp}")

            conn = get_db_connection()
            cur = conn.cursor()

            # Mark as 'running' so the client sees progress
            cur.execute(
                "UPDATE submissions SET status = 'running' WHERE id = %s",
                (submission_id,),
            )
            conn.commit()

            # Fetch the test cases for this problem
            cur.execute(
                "SELECT test_cases FROM problems WHERE id = %s",
                (job["problem_id"],),
            )
            test_cases = cur.fetchone()[0]

            # Simulate real execution time (sandboxes typically take 1-3 seconds)
            time.sleep(1.0)

            # Run each test case
            results = []
            all_passed = True
            for i, tc in enumerate(test_cases):
                result = run_test_case(job["code"], tc)
                results.append(result)
                if not result.get("passed", False):
                    all_passed = False
                icon = "✅" if result.get("passed") else "❌"
                print(f"[Worker]   Test {i+1}: {icon}")

            # Determine final status
            final_status = "accepted" if all_passed else "wrong_answer"

            # Update the database with results
            cur.execute("""
                UPDATE submissions
                SET status = %s, results = %s, runtime_ms = %s, memory_kb = %s
                WHERE id = %s
            """, (final_status, json.dumps(results), 42, 15200, submission_id))
            conn.commit()

            emoji = '🎉' if all_passed else '💔'
            print(f"[Worker] {emoji} Submission {submission_id} → {final_status}")

            cur.close()
            conn.close()
            job_queue.task_done()

        except Exception as e:
            print(f"[Worker] ❌ Error: {e}")


# Start the worker in a background thread
worker_thread = threading.Thread(target=worker, daemon=True, name="submission-worker")
worker_thread.start()
print("✅ Worker thread started — listening for jobs on the queue")


## 🔄 Step 4: Polling for Results

Now that we have the API (submit), queue, and worker, there’s one piece missing:
**how does the client know when the results are ready?**

The answer is **polling** — the client repeatedly calls `GET /submissions/:id`
every ~1 second until the status changes from `pending`/`running` to a final
status like `accepted` or `wrong_answer`.

**Try this yourself:** Open LeetCode, open your browser’s Network tab, and
submit a solution. You’ll see repeated requests to the submissions endpoint!

Let’s implement our own polling function:


In [ ]:
# === 🔄 Polling Function (simulates what the frontend does) ===


def poll_submission(submission_id, max_wait=20):
    """
    Poll GET /submissions/:id every second until we get a final result.
    This is exactly what the LeetCode frontend does.
    """
    print(f"[Client] 🔍 Polling submission {submission_id}...")

    last_status = None
    start = time.time()

    while time.time() - start < max_wait:
        conn = get_db_connection()
        cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
        cur.execute(
            "SELECT id, status, results, runtime_ms, memory_kb FROM submissions WHERE id = %s",
            (submission_id,),
        )
        sub = cur.fetchone()
        cur.close()
        conn.close()

        # Only print when the status changes
        if sub['status'] != last_status:
            timestamp = time.strftime('%H:%M:%S')
            print(f"[Client] ⏱️  [{timestamp}] Status: {sub['status']}")
            last_status = sub['status']

        # If status is final (not pending or running), we're done!
        if sub['status'] not in ('pending', 'running'):
            print(f"\n[Client] 🏁 Final result: {sub['status']}")
            if sub['results']:
                for i, r in enumerate(sub['results']):
                    icon = "✅" if r.get('passed') else "❌"
                    output = r.get('output', r.get('error', 'N/A'))
                    print(f"   Test {i+1}: {icon} Output={output}")
            if sub['runtime_ms']:
                print(f"   Runtime: {sub['runtime_ms']}ms | Memory: {sub['memory_kb']}KB")
            return sub

        time.sleep(1)  # Wait 1 second before polling again

    print("[Client] ⏰ Polling timed out!")
    return None


# --- Demo: Submit a solution through the full pipeline and poll ---
print("Submitting a correct Two Sum solution...\n")

sub_id = submit_code(
    user_id=1,
    problem_id=1,
    language="python",
    code="""
class Solution:
    def twoSum(self, nums, target):
        seen = {}
        for i, num in enumerate(nums):
            complement = target - num
            if complement in seen:
                return [seen[complement], i]
            seen[num] = i
""",
)
notebook_submission_ids.append(sub_id)

time.sleep(0.3)  # Brief pause so output is nicely ordered
poll_submission(sub_id)


### 💡 Alternatives to Polling

Polling isn’t the only option. Here are some alternatives and why LeetCode
likely chose polling:

| Approach | How It Works | Pros | Cons |
|----------|-------------|------|------|
| **Polling** | Client asks every ~1s | Simple, reliable, works everywhere | Slightly wasteful requests |
| **WebSockets** | Server pushes when ready | Real-time, no wasted requests | Complex to implement, connection management |
| **Server-Sent Events** | Server pushes via HTTP stream | Simpler than WebSockets | One-directional only |
| **Long Polling** | Client waits, server holds connection | Fewer requests than polling | Ties up server connections |

**Why polling wins here:** Code execution takes 1–5 seconds. Making 2–5 extra HTTP
requests is trivial. The simplicity of polling far outweighs the minor overhead.
WebSockets would be overkill for a “wait 3 seconds” use case.


## 🔬 Step 5: Putting It All Together

Let’s run the **full pipeline end-to-end** with two submissions:
- A **correct** solution (should get `accepted`)
- An **incorrect** solution (should get `wrong_answer`)

Watch the timestamps — you’ll see the async nature of the system.
The API returns immediately while the worker processes in the background.


In [ ]:
# === 🔬 Full End-to-End Demo ===

print("=" * 60)
print("🎬 End-to-End Demo: Correct vs. Incorrect Submissions")
print("=" * 60)

# 1. Pick a problem
print("\n1️⃣  Problem: Two Sum (Problem #1)")
print("   Given nums and a target, return indices that add to target.\n")

# 2. Submit a CORRECT solution
print("2️⃣  Submitting CORRECT solution...")
correct_id = submit_code(
    user_id=2,
    problem_id=1,
    language="python",
    code="""
class Solution:
    def twoSum(self, nums, target):
        seen = {}
        for i, num in enumerate(nums):
            complement = target - num
            if complement in seen:
                return [seen[complement], i]
            seen[num] = i
""",
)
notebook_submission_ids.append(correct_id)

# 3. Submit an INCORRECT solution
print("\n3️⃣  Submitting INCORRECT solution...")
wrong_id = submit_code(
    user_id=2,
    problem_id=1,
    language="python",
    code="""
class Solution:
    def twoSum(self, nums, target):
        return [0, 0]  # Always returns [0, 0] — wrong!
""",
)
notebook_submission_ids.append(wrong_id)

# 4. Poll both submissions (worker processes them one at a time)
print("\n4️⃣  Polling for results...")
print("-" * 40)
time.sleep(0.5)

print("\n--- Correct submission ---")
poll_submission(correct_id)

print("\n--- Incorrect submission ---")
poll_submission(wrong_id)


## 📊 Step 6: Why This Architecture Matters at Scale

During a LeetCode Weekly Contest, **100,000+ users** submit solutions within
the same 90-minute window. That means **thousands of submissions per second**
at peak.

Without a queue, the API server would try to execute all of them at once
and quickly run out of resources (CPU, memory, threads).

With the queue:
1. The API server stays fast — it just inserts a row and pushes to the queue
2. The queue **buffers** the burst — it can hold millions of messages
3. Workers **drain** the queue at their own pace
4. Need more throughput? **Add more workers** (horizontal scaling)

Let’s simulate a burst:


In [ ]:
# === 📊 Simulating a Contest Burst ===

import random

print("🏁 Simulating contest burst: 10 submissions arriving at once!\n")

correct_code = """
class Solution:
    def twoSum(self, nums, target):
        seen = {}
        for i, num in enumerate(nums):
            complement = target - num
            if complement in seen:
                return [seen[complement], i]
            seen[num] = i
"""

wrong_code = """
class Solution:
    def twoSum(self, nums, target):
        return [0, 0]
"""

burst_ids = []
start = time.time()

# Submit 10 solutions rapidly (simulating many users at contest start)
for i in range(10):
    code = correct_code if i < 4 else wrong_code  # 4 correct, 6 wrong
    sub_id = submit_code(
        user_id=(i % 50) + 1,
        problem_id=1,
        language="python",
        code=code,
    )
    burst_ids.append(sub_id)
    notebook_submission_ids.append(sub_id)

enqueue_time = time.time() - start
print(f"\n📨 All 10 submissions queued in {enqueue_time:.3f}s!")
print(f"📦 Queue depth right now: ~{job_queue.qsize()} jobs")
print("   (The worker is draining the queue one by one...)\n")

# Wait for all jobs to be processed
job_queue.join()

total_time = time.time() - start

# Check final results
conn = get_db_connection()
cur = conn.cursor()
cur.execute("""
    SELECT status, COUNT(*) FROM submissions
    WHERE id = ANY(%s)
    GROUP BY status
    ORDER BY status
""", (burst_ids,))
results = cur.fetchall()
cur.close()
conn.close()

print("\n📊 Results Summary:")
print("-" * 30)
for status, count in results:
    icon = "✅" if status == "accepted" else "❌"
    print(f"  {icon} {status}: {count}")
print(f"\n⏱️  Total time: {total_time:.1f}s for 10 submissions (1 worker)")
print(f"   With 10 workers in parallel: ~{total_time / 10:.1f}s")


### 🔑 Key Insight: Workers Scale Horizontally

In our demo, one worker processed all 10 submissions **sequentially** (~10 seconds).
In production, you’d run **many workers in parallel**:

```
                          ┌── Worker 1 ──→ Sandbox 1
                          ├── Worker 2 ──→ Sandbox 2
  API ──→ Queue (SQS) ───├── Worker 3 ──→ Sandbox 3
                          ├── ...         ...
                          └── Worker N ──→ Sandbox N
```

Each worker pulls jobs from the same queue independently. With 100 workers,
you can process 100 submissions simultaneously. During a contest, auto-scaling
can spin up hundreds of worker containers to handle the load.

This is the beauty of the **queue + worker** pattern — adding capacity is as
simple as launching more containers.


## 🧹 Cleanup

Let’s clean up the submissions we created during this notebook so the
database stays in its original state.


In [ ]:
# === 🧹 Cleanup ===

# Stop the worker thread
worker_running = False
worker_thread.join(timeout=5)
print("✅ Worker thread stopped")

# Delete submissions we created during this notebook
if notebook_submission_ids:
    conn = get_db_connection()
    cur = conn.cursor()

    # Delete competition_submissions that reference our submissions first (FK constraint)
    cur.execute(
        "DELETE FROM competition_submissions WHERE submission_id = ANY(%s)",
        (notebook_submission_ids,),
    )

    # Delete our submissions
    cur.execute(
        "DELETE FROM submissions WHERE id = ANY(%s)",
        (notebook_submission_ids,),
    )
    deleted = cur.rowcount
    conn.commit()
    cur.close()
    conn.close()
    print(f"✅ Cleaned up {deleted} submissions created during this notebook")

# Clean up any Redis keys we may have set
r = get_redis_client()
for key in r.scan_iter("notebook:*"):
    r.delete(key)
print("✅ Redis keys cleaned up")

print("\n🏁 All clean! You can re-run this notebook from the top.")


## 📚 Summary

In this notebook, we traced the full journey of a LeetCode code submission:

```
User clicks Submit
       │
       ▼
API creates submission (status='pending')
       │
       ▼
Job placed on queue
       │
       ▼
Worker picks up job (status='running')
       │
       ▼
Code executed against test cases
       │
       ▼
Results saved to DB (status='accepted' or 'wrong_answer')
       │
       ▼
Client polling detects final status → shows results
```

### 🔑 Key Takeaways

1. **Submissions go through a lifecycle:** `pending → running → accepted / wrong_answer / runtime_error / time_limit`

2. **Code execution is async** — the API returns immediately with a submission ID.
   It never runs code itself.

3. **A queue decouples API from workers** — this enables buffering during traffic
   spikes, retries on failure, and independent scaling of workers.

4. **Clients poll for results** — simple, effective, and exactly what real LeetCode
   uses. No need for WebSockets when the wait is 1–5 seconds.

---

### ➡️ Next Up

In **Notebook 2: Sandboxed Code Execution**, we’ll look inside the sandbox
container and understand how untrusted code runs safely — with read-only
filesystems, no network access, memory limits, and CPU constraints.
